In [1]:
# for working part
import numpy as np
import pandas as pd


# for eda
from ydata_profiling import ProfileReport

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_22008\3133939372.py:7: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
# reading the parquet file created in notebook 2

lead_conversion_data = pd.read_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\raw_data.parquet")

In [3]:
# removing null rows of assigned date as it is not needed

lead_conversion_data  =  lead_conversion_data[~lead_conversion_data['assigned_date'].isnull()]

In [4]:
lead_conversion_data.columns

Index(['lead_id_x', 'pii_id', 'owner', 'lead_source', 'assigned_date',
       'follow_up_date', 'contact_id', 'lead_id_y', 'profile', 'country',
       'state', 'total_duration', 'call_count', 'last_call_date',
       'first_call_date', 'distinct_call_days', 'connected_call_count',
       'missed_call_count', 'inbound_call_count', 'outbound_call_count',
       'converted'],
      dtype='object')

In [5]:
# extracting month and year from raw assigned date 

lead_conversion_data['assigned_month'] = lead_conversion_data['assigned_date'].dt.month
lead_conversion_data['assigned_year'] = lead_conversion_data['assigned_date'].dt.year

In [6]:
# followup done column
lead_conversion_data['followup_done'] = np.where(
    (lead_conversion_data['follow_up_date'].notna()) & (lead_conversion_data['last_call_date'] > lead_conversion_data['follow_up_date']),
     1,0 )

In [8]:
# adding new features from old columns


lead_conversion_data['average_duration'] = lead_conversion_data['total_duration']/lead_conversion_data['call_count']
lead_conversion_data['connection_rate'] = lead_conversion_data['connected_call_count'] / lead_conversion_data['call_count']
lead_conversion_data['miss_rate'] = lead_conversion_data['missed_call_count'] / lead_conversion_data['call_count']
lead_conversion_data['average_call_per_day'] = lead_conversion_data['call_count'] / lead_conversion_data['distinct_call_days']
lead_conversion_data['average_duration_per_day'] = lead_conversion_data['total_duration'] / lead_conversion_data['distinct_call_days']
lead_conversion_data['inbound_outbound_ratio'] = lead_conversion_data['inbound_call_count'] / lead_conversion_data['outbound_call_count']
lead_conversion_data['time_taken_for_first_touch'] = (lead_conversion_data['first_call_date'] - lead_conversion_data['assigned_date']).dt.days
lead_conversion_data['call_span_days'] = (lead_conversion_data['last_call_date'] - lead_conversion_data['first_call_date']).dt.days
lead_conversion_data['call_frequency'] = (lead_conversion_data['call_count'] / (lead_conversion_data["call_span_days"] + 1 ))

In [9]:
# removing not important columns for the model
lead_conversion_data = lead_conversion_data[
    [ 'owner', 'lead_source','profile', 'total_duration', 'call_count', 'distinct_call_days', 'connected_call_count',
       'missed_call_count', 'inbound_call_count', 'outbound_call_count', 'converted', 'assigned_month', 'assigned_year',
       'followup_done', 'average_duration', 'connection_rate', 'miss_rate', 'average_call_per_day', 'average_duration_per_day',
       'inbound_outbound_ratio', 'time_taken_for_first_touch', 'call_span_days', 'call_frequency']]

In [ ]:
# will do EDA again on processed data

In [10]:
# finally loading the processed dataset into parquet file


lead_conversion_data.to_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\processed_data.parquet",engine="fastparquet",index=False)